Data Extraction and Preprocesing with CIAO and Chandra LC

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.stats import bayesian_blocks
import glob
import tqdm
import os
import csv

In [ ]:
# CHANGING DIRECTORIES EMILLYYYYY
## Path to directory holding all sources for a specific galaxy
all_directory = "D:\Training_Sources2\Training_Sources" # Directory where all sources are

binsize = 500

# Do Not Touch

source_directories = sorted(glob.glob(f'{all_directory}\\2CXO_J002400*\\')) # Where all the sources directories are


print(source_directories)

In [ ]:
for emilyaleman in source_directories:
    os.chdir(emilyaleman)
    tar = glob.glob(emilyaleman + "*.tar")
    if tar:
        source_tar = tar[0]
    else:
        print("No matching directory found.")
    os.system("tar -xvf "+ source_tar)
    matching_directories = glob.glob(emilyaleman + "/" + "cdapackage*")
    if matching_directories:
        target_directory = matching_directories[0]
        os.chdir(target_directory)
        print("Changed to directory:", target_directory)
    else:
        print(f"No matching directory found for {emilyaleman}")
    os.system("gunzip *")


    cmd = "lc_filemaker.sh " + target_directory + "/ " + str(binsize)

    print(cmd)
    os.system(cmd)

stop
try:
    hrc_txt_files = glob.glob(target_directory + "/hrc*.txt")
    hrc_csv_files = glob.glob(target_directory + "/hrc*.csv")

    hrc_files = hrc_txt_files + hrc_csv_files
    for i in hrc_files:
        print(f"-------------------- Deleting: {i} --------------------")
        os.system("rm " + i)
except Exception as e:
    print(f"{e}")
try:
    acis_files_bin = glob.glob(target_directory + "/acis*bintime.txt")
    acis_files_txt = glob.glob(target_directory + "/acis*.lc.txt")
    acis_files_csv = glob.glob(target_directory + "/acis*.csv")
    acis_files = acis_files_csv+acis_files_txt
    available = []
    for j in acis_files_bin:
        available.append(j[-17:-12])
    print(available)
    for k in acis_files:
        file_id = k[-41:-36]
        if  file_id not in available:
            print(f"-------------------- Deleting: {k} --------------------")
            os.system("rm " + k)
except Exception as e:
    print(f"{e}")
try:
    available2 = []
    for m in acis_files_txt:
        available2.append(m[-41:-36])
    print(available2)
    for l in acis_files_bin:
        file_id = l[-17:-12]
        if  file_id not in available2:
            print(f"-------------------- Deleting: {l} --------------------")
            os.system("rm " + l)
except Exception as e:
    continue

In [ ]:
source_directories = sorted(glob.glob(f'{all_directory}/2CXO*/'))
print(source_directories)

for emilyaleman in source_directories:
    os.chdir(emilyaleman)
    matching_directories = glob.glob('cdapackage*')
    if matching_directories:
        os.system("rm -rf " + matching_directories[0])
        print(matching_directories[0])
    else:
        continue
os.chdir(all_directory)
matching_directories = glob.glob('PLOTS')
if matching_directories:
        os.system("rm -rf " + matching_directories[0])
        print(matching_directories[0])

In [ ]:
bad_obsid = []
bad_data = []
bad_count = []
for sources in tqdm.tqdm(source_directories):
    directory_name = os.path.basename(sources.rstrip('/'))
    new_directory = f"PLT_{directory_name}"
    image_directory = all_directory + "PLOTS" + "/"
    new_directory_path = image_directory + new_directory + "/"
    os.makedirs(new_directory_path, exist_ok=True)
    matching_directories = glob.glob(sources  + "cdapackage*")
    if matching_directories:
        target_directory = matching_directories[0]
        os.chdir(target_directory)
        # print("Changed to directory:", target_directory)
    else:
        # print(f"No matching directory found for {sources}")
        continue
    directory_name = os.path.basename(sources.rstrip('/'))
    broad_files = sorted(glob.glob(f'{target_directory}/*b.lc.txt'))
    soft_files = sorted(glob.glob(f'{target_directory}/*s.lc.txt'))
    medium_files = sorted(glob.glob(f'{target_directory}/*m.lc.txt'))
    hard_files = sorted(glob.glob(f'{target_directory}/*h.lc.txt'))
    bintime_files = sorted(glob.glob(f'{target_directory}/*bintime.txt'))
    try:

        all_files = np.stack((broad_files, soft_files, medium_files, hard_files, bintime_files), axis=1)
        # print(directory_name)
        durations = []

        for file in all_files:
            broad_data = np.loadtxt(file[0])
            soft_data = np.loadtxt(file[1])
            medium_data = np.loadtxt(file[2])
            hard_data = np.loadtxt(file[3])

            obsid = file[0][-41:-36]
            regid = file[0][-26:-22]

            count = 0
            if broad_data.size == 0:
                count += 1

            if count  == 1:
                bad_obsid.append(f"{directory_name}: {obsid}")
            else:
                max_count = broad_data[:,1].max()
                if max_count > .01:
                    data_stack = np.stack((broad_data, soft_data, medium_data, hard_data), axis=1)

                    filtered = []
                    for idx, data_stack_i in enumerate(data_stack):
                        array = data_stack[idx]
                        if array[0][4] != 0:
                            filtered.append(array)
                    filtered_new = np.array(filtered)

                    bd = filtered_new[:,0]

                    min_time = bd[:,0].min()
                    max_time = bd[:,0].max()
                    durations.append(max_time - min_time)
                else:
                    bad_count.append(f"{directory_name}: {obsid}")

        if sum(durations) == 0:
            # print(f"Skipping Source {directory_name}: No Good Observations")
            continue
        max_duration = max(durations)/1000

        for file in all_files:
            broad_data = np.loadtxt(file[0])

            obsid = file[0][-41:-36]


            check_obsid = f"{directory_name}: {obsid}"
            if check_obsid in bad_obsid:
                # print(f"Skipping {directory_name}:{obsid} -> HRC")
                continue
            else:
                if check_obsid in bad_data:
                    # print(f"Skipping {directory_name}:{obsid} -> No Data")
                    continue
                else:
                    broad_data = np.loadtxt(file[0])
                    soft_data = np.loadtxt(file[1])
                    medium_data = np.loadtxt(file[2])
                    hard_data = np.loadtxt(file[3])
                    bintime = np.loadtxt(file[4])

                    if check_obsid not in bad_count:

                        data_stack = np.stack((broad_data, soft_data, medium_data, hard_data), axis=1)

                        filtered = []
                        for idx, data_stack_i in enumerate(data_stack):
                            array = data_stack[idx]
                            if array[0][4] != 0:
                                filtered.append(array)
                        filtered_new = np.array(filtered)

                        bd = filtered_new[:,0]
                        print(bd)
                        sd = filtered_new[:,1]
                        md = filtered_new[:,2]
                        hd = filtered_new[:,3]

                        min_time = bd[:,0].min()
                        time_data = (bd[:,0] - min_time)/1000


                        current = []
                        current_count = 0
                        for val in bd[:,3]:
                            current_count += val
                            current.append(current_count)



                        obsid = file[0][-41:-36]
                        regid = file[0][-26:-22]
                        soft_counts = sd[:,3]
                        medium_counts = md[:,3]
                        hard_counts = hd[:,3]

                        total_counts = soft_counts+medium_counts+hard_counts


                        small_constant = 1e-6
                        hardness = (medium_counts - soft_counts) / (total_counts + small_constant)
                        df = pd.DataFrame({'Time': time_data, 'Hardness': hardness})



                        fig = plt.figure(figsize=(12, 10)) # Define the figure

                        gs = fig.add_gridspec(3, 2) # Define the grid layout. We now have 4 rows and 2 columns

                        axs0 = fig.add_subplot(gs[0, :]) # Add first subplot in grid. It spans across both columns.
                        axs1 = fig.add_subplot(gs[1, :]) # Add second subplot in grid. It spans across both columns.
                        axs2 = fig.add_subplot(gs[2, 0]) # Add third subplot in grid in the left column of the third row.
                        axs3 = fig.add_subplot(gs[2, 1]) # Add fourth subplot in grid in the right column of the third row.

                        # Now you can plot using the individual axes
                        axs0.plot(time_data, bd[:,1], color = 'blue', label='Broad')
                        axs1.plot(time_data, sd[:,1], color='red', label='Soft')
                        axs1.plot(time_data, md[:,1], color='green', label='Medium')
                        axs1.plot(time_data, hd[:,1], color = 'blue', label='Hard')
                        axs2.plot(time_data, current, color = 'black', label='Cumulative Count')
                        axs3.plot(time_data, hardness, color = 'purple', label='Hardness Ratio')

                        max_y = max(abs(hardness.min()), abs(hardness.max()))

                        # Set the y-axis limits to be symmetrical around zero
                        axs3.set_ylim(-max_y, max_y)

                        # Calculate and plot moving average of hardness ratio
                        window_size = 20  # Window size for moving average
                        hardness_df = pd.DataFrame({'Time': time_data, 'Hardness': hardness})
                        hardness_df['Moving Average'] = hardness_df['Hardness'].rolling(window_size,min_periods=5).mean()
                        axs3.plot(hardness_df['Time'], hardness_df['Moving Average'], color='orange', label='Moving Average')

                        # Setting labels for each subplot
                        axs0.set_ylabel('Broad Counts', fontsize=10)
                        axs1.set_ylabel('H-M-S Counts', fontsize=10)
                        axs2.set_ylabel('Cumulative Counts', fontsize=10)
                        axs3.set_ylabel('$(M-S)/T$', fontsize=10)  # LaTeX rendered text

                        axs0.tick_params(labelsize=14)  # Change the fontsize of tick labels
                        axs1.tick_params(labelsize=14)  # Change the fontsize of tick labels
                        axs2.tick_params(labelsize=14)  # Change the fontsize of tick labels
                        axs3.tick_params(labelsize=14)  # Change the fontsize of tick labels

                        axs0.set_xlim([0, max_duration])
                        axs1.set_xlim([0, max_duration])
                        axs2.set_xlim([0, max_duration])
                        axs3.set_xlim([0, max_duration])

                        fig.supxlabel('Time (ks)',fontsize=15)

                        axs0.legend()
                        axs1.legend()
                        axs2.legend()
                        axs3.legend(loc='upper right')

                        fig.supylabel('Count Rate (Counts/Second)',fontsize=15)
                        fig.suptitle(f'Source: {directory_name}: ObsID {obsid} with ~{binsize}s Bin size',fontsize=20)

                        os.chdir(new_directory_path)
                        fig.savefig(f"{directory_name}_{obsid}_{regid}.png")
                    else:
                        # print(f"Skipping {directory_name}:{obsid} -> Low Count Rate <.01")
                        continue
    except ValueError:
        print(f"Skipping {directory_name}: VALUE ERROR")

Bayesian Block and Event List Generator

In [ ]:
#GENERATES EVENT SOURCE FILES

import os
import glob
from astropy.io import fits
from astropy.stats import bayesian_blocks
import matplotlib.pyplot as plt
import csv

threshold_percentage = .12

#NEW DATA
all_directory = "D:\\Training_Sources"
input_directory = sorted(glob.glob(f"{all_directory}\\2CXO_J002400*\\cdapackage*\\*regevt3.fits"))
print(input_directory)

#OLD DATA
#all_directory = "D:\Training_Sources"
output_directory = os.path.join(all_directory, "CSV_data")

os.makedirs(output_directory, exist_ok=True)  #Create the "CSV_data" directory if it doesn't exist

for source in input_directory:
    obsID = os.path.basename(source).split('_')[0]

    parts = source.split("\\")
    if len(parts) >= 2:
        directory_name = parts[-3]
    else:
        print(f"Cannot extract directory name from path: {source}")
        continue

    # Create a subdirectory for each source within the "CSV_data" directory
    source_directory = os.path.join(output_directory, f"{directory_name}")
    os.makedirs(source_directory, exist_ok=True)  # Create the source directory if it doesn't exist

    # Create a sub-subdirectory for each obsID within the source directory
    obs_directory = os.path.join(source_directory, f"{obsID}")
    os.makedirs(obs_directory, exist_ok=True)  # Create the obsID directory if it doesn't exist

    with fits.open(source) as hdul:
        event_data = hdul['EVENTS'].data
        time_column = event_data['TIME']


    event_source = []

    # Use Astropy's Bayesian block algorithm
    blocks = bayesian_blocks(time_column, fitness='events')

    # Initialize a list to store window intervals
    windows = []

    # Iterate through the blocks to establish windows
    for i in range(len(blocks) - 1):
        start_time = blocks[i]
        end_time = blocks[i + 1]
        windows.append((start_time, end_time))
        print(windows)

    total_data_count = len(time_column)

    # Determine if data in each window falls within the threshold percentage
    for window in windows:
        start_time, end_time = window

        # Filter data within the window
        data_in_window = time_column[(time_column >= start_time) & (time_column < end_time)]
        data_count_in_window = len(data_in_window)

        # Calculate the percentage of data within the window
        percentage_in_window = (data_count_in_window / total_data_count) * 100

        if percentage_in_window >= threshold_percentage:
            print(f"ObsID: {obsID}, Window: ({start_time}, {end_time}) contains {percentage_in_window:.2f}% of the data.")
        else:
            print(f"ObsID: {obsID}, Window: ({start_time}, {end_time}) does not meet the threshold percentage.")

    event_source.extend(event_data.flatten().tolist())

    # Create a histogram to visualize the Bayesian blocks
    plt.hist(time_column, bins=blocks, histtype='step', color='blue', label='Bayesian Blocks', density=True)
    plt.xlabel('Time (s)')
    plt.ylabel('Counts')

    # Overplot the original data
    plt.hist(time_column, bins=100, color='gray', alpha=0.9, label='Real Data', density=True)

    title = f'Bayesian Blocks Source: {directory_name} ObsID: {obsID}'

    for block_edge in blocks:
        plt.axvline(block_edge, color='red', linestyle='--', linewidth=1)

    plt.legend(loc='upper right')
    plt.title(title)
    plt.show()


    # Save time_source and event_source to CSV in the specific obsID subdirectory
    event_csv_path = os.path.join(obs_directory, f"event_source_{obsID}.csv")

    with open(event_csv_path, 'w', newline='') as event_csvfile:
        event_writer = csv.writer(event_csvfile)
        event_writer.writerows(event_source)

Feature Engineering

In [ ]:
import os
import glob
from astropy.io import fits
from astropy.stats import bayesian_blocks
import csv
import pandas as pd

threshold_percentage = .12

#NEW DATA
all_directory = "D:\Training_Sources"
input_directory = sorted(glob.glob(f"{all_directory}\\2CXO_*\\cdapackage*\\*regevt3.fits"))

# OLD DATA
output_directory = os.path.join(all_directory, "CSV_data")

os.makedirs(output_directory, exist_ok=True)  # Create the "CSV_data" directory if it doesn't exist

for source in input_directory:
    obsID = os.path.basename(source).split('_')[0]

    parts = source.split("\\")
    if len(parts) >= 2:
        directory_name = parts[-3]
    else:
        print(f"Cannot extract directory name from path: {source}")
        continue

    # Create a subdirectory for each source within the "CSV_data" directory
    source_directory = os.path.join(output_directory, f"{directory_name}")
    os.makedirs(source_directory, exist_ok=True)  # Create the source directory if it doesn't exist

    # Create a sub-subdirectory for each obsID within the source directory
    obs_directory = os.path.join(source_directory, f"{obsID}")
    os.makedirs(obs_directory, exist_ok=True)  # Create the obsID directory if it doesn't exist

    print(f"Saving CSV files to: {obs_directory}")

    with fits.open(source) as hdul:
        event_data = hdul['EVENTS'].data
        time_column = event_data['TIME']

    # Use Astropy's Bayesian block algorithm
    blocks = bayesian_blocks(time_column, fitness='events')

    # Initialize a list to store window intervals
    windows = []

    # Iterate through the blocks to establish windows
    for i in range(len(blocks) - 1):
        start_time = blocks[i]
        end_time = blocks[i + 1]
        windows.append((start_time, end_time))

    total_data_count = len(time_column)

    # Initialize a list to store event data within each window
    event_source = []

    # Initialize a list to store window features
    window_features = []

    # Process each Bayesian block window
    for i in range(len(blocks) - 1):
        # Define start and end of the window
        start_time = blocks[i]
        end_time = blocks[i + 1]

        # Filter data within the window
        data_in_window = time_column[(time_column >= start_time) & (time_column < end_time)]

        # Count the number of events in the window
        count_in_window = len(data_in_window)

        # Calculate the duration of the window
        duration = end_time - start_time

        # Calculate the count rate for the window, handling division by zero
        count_rate = count_in_window / duration if duration > 0 else 0

        # Calculate the percentage of data within the window
        percentage_in_window = (count_in_window / total_data_count) * 100

        # Append features to the list
        window_features.append([start_time, end_time, duration, count_in_window, count_rate, percentage_in_window])

    # Create a DataFrame from the list of features
    column_names = ['start_time', 'end_time', 'duration', 'count', 'count_rate', 'percentage_in_window']
    df = pd.DataFrame(window_features, columns=column_names)

    # Save the dataset to a CSV file
    csv_path = os.path.join(obs_directory, f'bayesian_blocks_dataset_{obsID}.csv')
    print(f"Saving CSV file to: {csv_path}")
    df.to_csv(csv_path, index=False)

Random Forest

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import confusion_matrix, classification_report


# Specify the directories for your CSV files
directories = [
    r"D:\Synthetic_2CXO\1.csv",
    r"D:\Synthetic_2CXO\1.1.csv",
    r"D:\Synthetic_2CXO\1.2.csv",
    r"D:\Synthetic_2CXO\2.csv",
    r"D:\Synthetic_2CXO\2.1.csv",
    r"D:\Synthetic_2CXO\2.2.csv",
    r"D:\Synthetic_2CXO\2.3.csv",
    r"D:\Synthetic_2CXO\2.4.csv",
    r"D:\Synthetic_2CXO\2.5.csv",
    r"D:\Training_Sources*\CSV_data\2CXO*\acisf*\bayesian_blocks_dataset*"
]

# Initialize an empty DataFrame to store all data
all_data = pd.DataFrame()

# Read each file and concatenate the data into a single DataFrame
for directory in directories:
    # Use a wildcard (*) to get all matching files
    files = glob.glob(directory)

    for file in files:
        df = pd.read_csv(file)  # Assuming your data is comma-separated by default
        all_data = pd.concat([all_data, df], ignore_index=True)

# Extract the features and target
X = all_data[['duration', 'count', 'count_rate', 'percentage_in_window']]
y = all_data['Label']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.41, random_state=42)

# Initialize the Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the training data
rf_classifier.fit(X_train, y_train)

# Make predictions on the test data
y_pred = rf_classifier.predict(X_test)

# Evaluate the performance of the model
accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:\n", classification_rep)


In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.datasets import make_classification

# Convert the classification report to a DataFrame
report_df = pd.DataFrame(classification_rep).transpose()

# Filter the DataFrame to include only Accuracy, Precision, Recall, F1-score, and Support
filtered_metrics = report_df[['precision', 'recall', 'f1-score']]

# Plot the heatmap
plt.figure(figsize=(10, 5))
heatmap = sns.heatmap(filtered_metrics.iloc[:-1, :].T, annot=True, cmap='Reds')
plt.title('Classification Report Heatmap')
plt.text(0.5, -0.12, f'Overall Accuracy: {accuracy: .3f}', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
plt.show()

# Use cross_val_score for cross-validation
cv_scores = cross_val_score(rf_classifier, X, y, cv=5)  # You can adjust the number of folds with the cv parameter

conf_matrix = confusion_matrix(y_test, y_pred)

# Visualize confusion matrix with seaborn heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Class 0', 'Class 1', 'Class 2'],
            yticklabels=['Class 0', 'Class 1', 'Class 2'],
            cbar_kws={'label': 'Count'})

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

#K-Folding

# Define the K-Fold Cross Validator
kfold = KFold(n_splits=5, random_state=42, shuffle=True)
kfold_scores = cross_val_score(rf_classifier, X, y, cv=kfold)

plt.figure(figsize=(10, 6))

# Plot accuracy for each fold
fold_labels = ['Fold {}: {:.4f}'.format(i+1, score) for i, score in enumerate(kfold_scores)]
for i, score in enumerate(kfold_scores):
    plt.plot(i+1, score, marker='o', linestyle='-', color='darkred')

# Plot line of best fit
z = np.polyfit(np.arange(1, len(kfold_scores)+1), kfold_scores, 15)
p = np.poly1d(z)
plt.plot(np.arange(1, len(kfold_scores)+1), p(np.arange(1, len(kfold_scores)+1)), "r--")

# Add labels and title
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('K-Fold Cross-Validation Scores')
plt.legend(fold_labels, loc='upper right')
plt.text(0.5, -0.13, f'Overall Cross-Validation Accuracy: {kfold_scores.mean(): .3f}', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
plt.grid(False)
plt.show()

# OOB vs Iterations

oob_errors = []
n_estimators_options = [15, 30, 60, 100, 200, 300]
legend_labels = []

for n_trees in n_estimators_options:
    rf = RandomForestClassifier(n_estimators=n_trees, oob_score=True, random_state=42)
    rf.fit(X_train, y_train)
    oob_error = 1 - rf.oob_score_
    oob_errors.append(oob_error)
    legend_labels.append(f"{n_trees} trees: OOB accuracy = {1 - oob_error:.3f}")

# Fit the line of best fit to the OOB errors
coefficients = np.polyfit(n_estimators_options, oob_errors, 15)
polynomial = np.poly1d(coefficients)
trendline = polynomial(n_estimators_options)

plt.figure(figsize=(10, 5))

# Plot the individual points
for i, n_trees in enumerate(n_estimators_options):
    plt.plot(n_trees, oob_errors[i], 'o', label=legend_labels[i], color='crimson')

# Plot the line of best fit
plt.plot(n_estimators_options, trendline, label='Trend Line', color='salmon')

plt.xlabel("Number of Trees")
plt.ylabel("OOB Error Rate")
plt.title("Out-of-Bag Error Rate vs. Number of Trees in Random Forest")
plt.legend()
plt.show()

In [ ]:
import glob
import pandas as pd

# Specify the directory for your new data CSV files
new_data_directory = r"D:\Validation_Set\CSV_data\2CXO_*\acisf*\bayesian_blocks_dataset_*"

# Use a wildcard (*) to get all matching files
new_data_files = glob.glob(new_data_directory)

# Initialize an empty list to store modified DataFrames
new_data_list = []

# Read each file, make predictions, and modify the DataFrame
for file in new_data_files:
    df = pd.read_csv(file)  # Assuming your data is comma-separated by default

    # Extract the features (assuming the 'Label' column is not present in new data)
    X_new = df[['duration', 'count', 'count_rate', 'percentage_in_window']]

    # Make predictions on the new data
    predictions_new = rf_classifier.predict(X_new)

    # Add the predictions to the current file DataFrame
    df['Label'] = predictions_new

    # Append the modified DataFrame to the list
    new_data_list.append(df)

# Concatenate the list of modified DataFrames
new_data = pd.concat(new_data_list, ignore_index=True)

# Print the predictions or save them to a new CSV file
print("Predictions on new data:\n", new_data[['duration', 'count', 'count_rate', 'percentage_in_window', 'Label']])

# Save the modified DataFrame to respective files
for i, file in enumerate(new_data_files):
    new_data_list[i].to_csv(file, index=False)


Energy Independence Testing

In [ ]:
import pandas as pd
from glob import glob
import os
from collections import defaultdict

def process_files(files):
    # Initialize an empty DataFrame
    combined_data = pd.DataFrame()

    # Process files and combine the second column ("COUNT_RATE")
    data_frames = []
    for file_path in files:
       # Skip the first line and use position-based column indexing
       data = pd.read_csv(file_path, delim_whitespace=True, skiprows=1, header=None, usecols=[1])
       data_frames.append(data)

    # Concatenate the data frames along the columns
    combined_data['COUNT_RATE'] = pd.concat(data_frames, axis=1).mean(axis=1)

    return combined_data

def process_observation(observation_directory, base_output_folder):
    # Extract the source ID from the observation directory
    source_id = os.path.basename(observation_directory)

    # Identify files for each observation and energy band
    all_files = sorted(glob(os.path.join(observation_directory, "cdapackage*", "acisf*regevt3.fits.*.lc.txt")))

    # Group files by observation IDs
    observation_groups = defaultdict(list)
    for file in all_files:
       observation_id = os.path.basename(file).split('_')[0]
       observation_groups[observation_id].append(file)

    # Process each group of files corresponding to the same observation
    for obs_id, files in observation_groups.items():
       # Group files by energy band
       energy_band_files = defaultdict(list)
       for file in files:
          energy_band = os.path.basename(file).split('.')[-3] # Extract energy band from the filename
          energy_band_files[energy_band].append(file)

       # Sort the energy bands to ensure consistent processing order
       sorted_energy_bands = sorted(energy_band_files.keys())

       # Initialize a variable to store the first column data
       time_data = None

       # Combine columns for each energy band
       combined_data = pd.DataFrame()
       for band in sorted_energy_bands:
          band_files = energy_band_files[band]
          band_data = process_files(band_files)
          combined_data[f'COUNT_RATE_{band}'] = band_data['COUNT_RATE']

            # If time_data is None, this is the first energy band being processed
          if time_data is None:
                time_data = pd.read_csv(band_files[0], delim_whitespace=True, skiprows=1, header=None, usecols=[0])

       # Add the first column from the first energy band
       if time_data is not None:
          combined_data['TIME'] = time_data.iloc[:,0]

       if not combined_data.empty:
            # Create the source directory within the base output folder if it doesn't exist
          source_directory = os.path.join(base_output_folder, source_id)
          os.makedirs(source_directory, exist_ok=True)

            # Create directory for the observation within the source directory
          obs_directory = os.path.join(source_directory, obs_id)
          os.makedirs(obs_directory, exist_ok=True)

            # Save the combined file in the observation directory
          output_filename = os.path.join(obs_directory, "merged_data.csv")
          combined_data.to_csv(output_filename, index=False)
       else:
            print(f"No data found for observation {obs_id}.")

# Specify the base directory
all_directory = "D:\Training_Sources"

# Specify the output folder
output_folder = os.path.join(all_directory, "Energy_Bands_CSV")

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Assuming all_directory contains subdirectories for each 2CXO source
source_directories = sorted(glob(os.path.join(all_directory, "2CXO*")))

# Process each source and save combined data to CSV
for source_directory in source_directories:
    process_observation(source_directory, output_folder)

In [ ]:
import glob
import pandas as pd
import os

# Define the directory containing the datasets
all_directory = r"D:\Training_Sources"

# Step 1: Find all files based on the new directory pattern
file_paths_A = sorted(glob.glob(os.path.join(all_directory, "CSV_data", "2CXO_*", "*", "bayesian_blocks_dataset_*")))
file_paths_B = sorted(glob.glob(os.path.join(all_directory, "Energy_Bands_CSV", "2CXO*", "*", "merged_*")))

# Step 2: Process Dataset A to find valid intervals and record them
all_intervals_A = {}

for file_path_A in file_paths_A:
    df_A = pd.read_csv(file_path_A)

    # Extract identifiers from the file path
    parts_A = file_path_A.split(os.sep)
    source_id_A = parts_A[-3]
    dataset_id_A = parts_A[-2]
    key_A = (source_id_A, dataset_id_A)

    # Find intervals based on the start and stop times along with labels
    intervals_A = list(zip(df_A.iloc[:, 0], df_A.iloc[:, 1], df_A.iloc[:, -1]))
    all_intervals_A[key_A] = intervals_A

# Step 3: Label the intervals in Dataset B
for file_path_B in file_paths_B:
    df_B = pd.read_csv(file_path_B)

    # Extract identifiers from the file path
    parts_B = file_path_B.split(os.sep)
    source_id_B = parts_B[-3]
    acisf_id_B = parts_B[-2]
    key_B = (source_id_B, acisf_id_B)

    # Initialize the Label column to 0
    df_B['Label'] = 0

    # If there's a matching key in Dataset A and there are intervals, apply the labels in Dataset B
    if key_B in all_intervals_A and all_intervals_A[key_B]:
        for start_A, stop_A, label_A in all_intervals_A[key_B]:
            df_B.loc[df_B['TIME'].between(start_A, stop_A, inclusive='both'), 'Label'] = label_A

    # Fill NaN values in the 'Label' column with 0
    df_B['Label'].fillna(0, inplace=True)

    # Save the modified Dataset B to the same CSV file
    df_B.to_csv(file_path_B, index=False)


In [ ]:
import os
import glob
import pandas as pd

# Set the directory path
all_directory = r"D:\Training_Sources"

# Get a list of file paths matching the specified pattern
files = sorted(glob.glob(os.path.join(all_directory, "Energy_Bands_CSV", "2CXO*", "*", "merged_*")))

# Define a function to calculate the mean for each energy band before, during, and after intervals labeled as 1 or 2
def calculate_means(file_path):
    # Read the CSV file into a pandas DataFrame
    df = pd.read_csv(file_path)

    # Extract columns 2, 3, 4, and 6
    energy_bands = df.iloc[:, 1:5]

    # Extract labels (column 6)
    labels = df.iloc[:, 5]

    # Check if there are any intervals labeled as 1 or 2
    if any(label in labels.values for label in [1, 2]):
        # Create a mask for intervals labeled as 1 or 2
        mask = labels.isin([1, 2])

        # Determine the candidate type
        candidate_type = "Planet Candidate (1)" if 1 in labels.values else "Eclipse Candidate (2)"

        # Calculate means for each energy band before and after intervals labeled as 1 or 2
        mean_before = energy_bands[~mask].mean()
        mean_after = energy_bands[mask].mean()

        # Calculate mean during intervals labeled as 1 or 2
        mean_during = energy_bands[mask].mean()

        # Print or store the results as needed
        print(f"File: {file_path}")
        print("Candidate Type:", candidate_type)
        print("Mean Before:", mean_before)
        print("Mean During:", mean_during)
        print("Mean After:", mean_after)
        print("="*50)

# Iterate over the files and calculate means for each
for file_path in files:
    calculate_means(file_path)

In [ ]:
import os
import glob
import pandas as pd
from scipy.stats import f_oneway

# Set the directory path
all_directory = r"D:\Training_Sources"

# Get a list of file paths matching the specified pattern
files = sorted(glob.glob(os.path.join(all_directory, "Energy_Bands_CSV", "2CXO_*", "*", "merged_*")))

# Define a function to perform ANOVA testing for energy bands during transit intervals
def perform_anova(file_path):
    # Read the CSV file into a pandas DataFrame
    df = pd.read_csv(file_path)

    # Extract columns 2, 3, and 4 for energy bands
    energy_bands = df.iloc[:, 1:4]

    # Extract labels (column 6)
    labels = df.iloc[:, 5]

    # Determine the candidate type
    candidate_type = "Planet Candidate" if 1 in labels.values else "Eclipse Candidate"

    # Check if there are any intervals labeled as 1 or 2
    if any(label in labels.values for label in [1, 2]):
        # Create a mask for intervals labeled as 1 or 2
        mask = labels.isin([1, 2])

        # Extract energy bands during transit intervals
        transit_data = energy_bands[mask]

        # Perform ANOVA test for energy bands during transit intervals
        f_statistic, p_value = f_oneway(*[transit_data.iloc[:, i] for i in range(transit_data.shape[1])])

        # Print or store the results as needed
        print(f"File: {file_path}")
        print("Candidate Type:", candidate_type)
        print("ANOVA F-Statistic:", f_statistic)
        print("ANOVA P-Value:", p_value)
        print("="*50)

        # You can add further analysis based on the p-value, such as checking if it's below a significance threshold
        if p_value < 0.05:
            print("The means of energy bands during transit intervals are significantly different.")
        else:
            print("The means of energy bands during transit intervals are not significantly different.")

# Iterate over the files and perform ANOVA testing for each
for file_path in files:
    perform_anova(file_path)